In [99]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from difflib import *

In [2]:
transformer = SentenceTransformer("all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7599.90it/s]


In [17]:
df = pd.read_csv("DataSet.csv")
df["allFeatures"] = df["genres"] + df["tag"]
df.drop(columns=["genres", "tag", "imdbId", "tmdbId"], inplace=True)
print(df.head())

   movieId                               title    rating  \
0        1                    Toy Story (1995)  3.921240   
1        2                      Jumanji (1995)  3.211977   
2        3             Grumpier Old Men (1995)  3.151040   
3        4            Waiting to Exhale (1995)  2.861393   
4        5  Father of the Bride Part II (1995)  3.064592   

                                         allFeatures  
0  Adventure|Animation|Children|Comedy|Fantasytoy...  
1  Adventure|Children|Fantasysee also:Zathura bad...  
2  Comedy|Romanceold moldy Jack Lemmon CLV sequel...  
3  Comedy|Drama|RomanceCLV characters revenge chi...  
4  ComedyCLV sequel fever Steve Martin sequel Dia...  


In [123]:
print(df.iloc[10743])

movieId                                                          59315
title                                                  Iron Man (2008)
rating                                                        3.913053
allFeatures          Action|Adventure|Sci-FiRDJ torture watch the c...
embedded_features    [0.044385727, 0.05323819, 0.02178637, -0.04523...
Name: 10743, dtype: object


In [121]:
movie_list = df["title"]
target = "iron man"
movie_name = get_close_matches(target, movie_list, cutoff=0.4)
movie_name = movie_name[0]
print(movie_name)

Iron Man (2008)


In [122]:
movie_index = df[df["title"] == movie_name].index[0]
print(movie_index)

10743


In [35]:
list = df["allFeatures"].values.tolist()
list_encoded = transformer.encode(list)
# print(list.flatten())

In [36]:
print(list)

["Adventure|Animation|Children|Comedy|Fantasytoy Animation Woody animation light buddy movie clever 3D Buzz Lightyear erlend's DVDs rated-G Want Tumey's To See Again 55 movies every kid should see--Entertainment Weekly Best of Rotten Tomatoes: All Time time travel action figures toys é˜®ä¸€é¸£ animated innovative witty disney kids and family story funny villian hurts toys soothing CLV warm BD-Video rousing kids movie humorous friendship computer animation cgi pixar DARING RESCUES Watched voice acting family film the boys Cartoon classic dolls buy imdb top 250 TÃ©a Leoni does not star in this movie adventure fun TOYS COME TO LIFE avi Disney cute Tumey's VHS Engaging fantasy first cgi film very good HEROIC MISSION fanciful UNLIKELY FRIENDSHIPS almost favorite action figure Disney animated feature lots of heart ya boy family Tom Hanks Tim Allen National Film Registry bright John Lasseter CGI CG animation USA 2009 reissue in Stereoscopic 3-D Pixar comedy Pixar animation children DVD-Video 

In [41]:
df["embedded_features"] = [row for row in list_encoded]

In [51]:
print(df.info)

<bound method DataFrame.info of        movieId                                          title    rating  \
0            1                               Toy Story (1995)  3.921240   
1            2                                 Jumanji (1995)  3.211977   
2            3                        Grumpier Old Men (1995)  3.151040   
3            4                       Waiting to Exhale (1995)  2.861393   
4            5             Father of the Bride Part II (1995)  3.064592   
...        ...                                            ...       ...   
19006   131054  Dinotopia: Quest for the Ruby Sunstone (2005)  2.000000   
19007   131082                              Playground (2009)  4.500000   
19008   131164                           Vietnam in HD (2011)  4.000000   
19009   131170                               Parallels (2015)  3.500000   
19010   131258                             The Pirates (2014)  2.500000   

                                             allFeatures  \
0      

In [46]:
from sklearn.metrics.pairwise import cosine_similarity

In [53]:
embedding_matrix = np.vstack(df["embedded_features"].values)
print(embedding_matrix)

[[ 0.03989261  0.01090856  0.03178258 ...  0.00705443 -0.02936784
   0.00893659]
 [ 0.02119415 -0.02861446 -0.01190723 ...  0.00826007  0.02335432
   0.02471898]
 [ 0.01564134  0.02228831  0.03616186 ...  0.01349148  0.0838851
   0.03090133]
 ...
 [ 0.02009066  0.0462891   0.01109822 ...  0.03313462 -0.02589702
  -0.02483381]
 [ 0.01576173 -0.00304485 -0.00773164 ...  0.04183321  0.00226553
  -0.00947392]
 [ 0.05809463  0.08204768 -0.02295801 ...  0.02346871 -0.01251449
   0.01619196]]


In [57]:
movie_similarity = cosine_similarity(embedding_matrix)

In [124]:
recoms = sorted(enumerate(movie_similarity[movie_index]), key=lambda x:x[1],reverse=True)
recoms = recoms[0:10]
print(recoms)

[(10743, np.float32(1.0000001)), (17846, np.float32(0.89386296)), (5544, np.float32(0.8924794)), (14058, np.float32(0.8817693)), (1267, np.float32(0.8752567)), (16103, np.float32(0.86643684)), (11545, np.float32(0.8630744)), (16053, np.float32(0.8622171)), (12659, np.float32(0.8610033)), (11541, np.float32(0.8606198))]


In [69]:
print(df["title"][0])

Toy Story (1995)


In [125]:
for recom in recoms:
    print(recom[0])

10743
17846
5544
14058
1267
16103
11545
16053
12659
11541


In [126]:
prinnt(df["title"])

NameError: name 'prinnt' is not defined

In [127]:
movies = [df["title"].iloc[recom] for recom in recoms]

IndexingError: Too many indexers

In [128]:
movies = [df["title"].iloc[recom[0]] for recom in recoms]
print(movies)

['Iron Man (2008)', 'Guardians of the Galaxy (2014)', 'Hulk (2003)', 'Captain America: The First Avenger (2011)', 'Batman & Robin (1997)', 'Iron Man 3 (2013)', 'Star Trek (2009)', 'Oblivion (2013)', 'Iron Man 2 (2010)', 'X-Men Origins: Wolverine (2009)']


In [64]:
print(sorted(movie_similarity[0], reverse=True)[:10])

[np.float32(0.9999999), np.float32(0.8986788), np.float32(0.8757764), np.float32(0.8680186), np.float32(0.8629863), np.float32(0.8608582), np.float32(0.82126963), np.float32(0.8208973), np.float32(0.81554365), np.float32(0.81426275)]


In [136]:
import numpy as np

# -----------------------------
# 1. FIX ALIGNMENT (CRITICAL)
# -----------------------------
df = df.reset_index(drop=True)
movieId_to_index = {mid: i for i, mid in enumerate(df["movieId"])}

embedding_matrix = np.array(embedding_matrix)
embedding_matrix = embedding_matrix / np.linalg.norm(embedding_matrix, axis=1, keepdims=True)

# -----------------------------
# 2. EVALUATION SETTINGS
# -----------------------------
K = 10
threshold = 4.0

precisions = []
recalls = []

user_groups = ratings.groupby("userId")

# -----------------------------
# 3. EVALUATION LOOP
# -----------------------------
for user_id, group in user_groups:

    liked_movies = group[group["rating"] >= threshold]["movieId"].values
    relevant_movies = set(liked_movies)

    if len(liked_movies) == 0:
        continue

    liked_indices = [movieId_to_index[m] for m in liked_movies if m in movieId_to_index]

    if len(liked_indices) == 0:
        continue

    # user profile vector
    user_vector = np.mean(embedding_matrix[liked_indices], axis=0)
    user_vector = user_vector / np.linalg.norm(user_vector)

    # FAST similarity (no sklearn)
    sims = np.dot(embedding_matrix, user_vector)

    # remove seen items
    sims[liked_indices] = -1

    # top-K recommendations
    rec_indices = np.argsort(sims)[-K:][::-1]
    recommended_movie_ids = df.iloc[rec_indices]["movieId"].values

    # -----------------------------
    # METRICS
    # -----------------------------
    hits = len(set(recommended_movie_ids) & relevant_movies)

    precisions.append(hits / K)
    recalls.append(hits / len(relevant_movies))

# -----------------------------
# 4. FINAL SCORES
# -----------------------------
print("Precision@10:", np.mean(precisions))
print("Recall@10:", np.mean(recalls))

Precision@10: 0.0
Recall@10: 0.0
